In [17]:
import pandas as pd
df = pd.read_excel("/Users/lzh/Downloads/NUS Y4S1 Downloads/BT2102/Assignment 3/data/raw/BeijingHousingPrices.xlsx", sheet_name=0)  

In [3]:
print(df.shape)           # (rows, columns)

(318851, 26)


In [6]:
print(df.head())  

                                                 url            id  \
0  https://bj.lianjia.com/chengjiao/101084782030....  101084782030   
1  https://bj.lianjia.com/chengjiao/101086012217....  101086012217   
2  https://bj.lianjia.com/chengjiao/101086041636....  101086041636   
3  https://bj.lianjia.com/chengjiao/101086406841....  101086406841   
4  https://bj.lianjia.com/chengjiao/101086920653....  101086920653   

          Lng        Lat            Cid  tradeTime     DOM  followers  \
0  116.475489  40.019520  1111027376244 2016-08-09  1464.0        106   
1  116.453917  39.881534  1111027381879 2016-07-28   903.0        126   
2  116.561978  39.877145  1111040862969 2016-12-11  1271.0         48   
3  116.438010  40.076114  1111043185817 2016-09-30   965.0        138   
4  116.428392  39.886229  1111027381174 2016-08-28   927.0        286   

   totalPrice  price  ...  buildingType  constructionTime renovationCondition  \
0       415.0  31680  ...           1.0              2005  

In [5]:
print(df.info())  

<class 'pandas.DataFrame'>
RangeIndex: 318851 entries, 0 to 318850
Data columns (total 26 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   url                  318851 non-null  str           
 1   id                   318851 non-null  object        
 2   Lng                  318851 non-null  float64       
 3   Lat                  318851 non-null  float64       
 4   Cid                  318851 non-null  int64         
 5   tradeTime            318851 non-null  datetime64[us]
 6   DOM                  160874 non-null  float64       
 7   followers            318851 non-null  int64         
 8   totalPrice           318851 non-null  float64       
 9   price                318851 non-null  int64         
 10  square               318851 non-null  float64       
 11  livingRoom           318819 non-null  float64       
 12  drawingRoom          318851 non-null  object        
 13  kitchen              3188

We expect livingRoom, drawingRoom, kitchen, bathRoom, buildingType, constructionTime, elevator, fiveYearsProperty, subway, communityAverage to all be the same type - int64. Having a float64/ object implies NaN values and/or having some values as text. Worth investigating.
Also worth checking if id is unique.

In [8]:
print(df.isna().sum())                      # missing values per column

url                         0
id                          0
Lng                         0
Lat                         0
Cid                         0
tradeTime                   0
DOM                    157977
followers                   0
totalPrice                  0
price                       0
square                      0
livingRoom                 32
drawingRoom                 0
kitchen                     0
bathRoom                    0
floor                       0
buildingType             2021
constructionTime            0
renovationCondition         0
buildingStructure           0
ladderRatio                 0
elevator                   32
fiveYearsProperty          32
subway                     32
district                    0
communityAverage          463
dtype: int64


Columns with NaN values to investigae are - DOM, livingRoom, buildingType, elevator, fiveYearsProperty, subway, communityAverage

In [10]:
cols = ["DOM", "livingRoom", "drawingRoom", "kitchen", "bathRoom", "buildingType",
        "constructionTime", "elevator", "fiveYearsProperty", "subway", "communityAverage"]

total = len(df)
rows = []
bad_mask = pd.DataFrame(index=df.index)        # one True/False column per variable

for c in cols:
    s = df[c]
    conv = pd.to_numeric(s, errors="coerce")

    is_missing = s.isna()
    is_text = conv.isna() & s.notna()
    is_frac = (conv % 1 != 0) & conv.notna()
    bad_mask[c] = is_missing | is_text | is_frac

    rows.append({
        "column": c,
        "dtype": s.dtype,
        "n_missing": is_missing.sum(),
        "n_text": is_text.sum(),
        "n_fractional": is_frac.sum(),
        "n_bad": bad_mask[c].sum(),
        "% missing": round(is_missing.mean() * 100, 3),
        "% text": round(is_text.mean() * 100, 3),
        "% fractional": round(is_frac.mean() * 100, 3),
        "% bad": round(bad_mask[c].mean() * 100, 3),
        "min": conv.min(),
        "max": conv.max(),
    })

summary = pd.DataFrame(rows).set_index("column")
display(summary)

# Rows with a problem in ANY of these columns (bad rows often overlap across columns)
rows_any = bad_mask.any(axis=1)
print(f"Rows with at least one problem: {rows_any.sum():,} ({rows_any.mean() * 100:.2f}% of {total:,})")

,dtype,n_missing,n_text,n_fractional,n_bad,% missing,% text,% fractional,% bad,min,max
column,,,,,,,,,,,
DOM,float64,157977,0,0,157977,49.546,0.000,0.00,49.546,1.000,1677.0
livingRoom,float64,32,0,0,32,0.010,0.000,0.00,0.010,0.000,9.0
drawingRoom,object,0,32,0,32,0.000,0.010,0.00,0.010,0.000,5.0
kitchen,int64,0,0,0,0,0.000,0.000,0.00,0.000,0.000,4.0
bathRoom,object,0,2,0,2,0.000,0.001,0.00,0.001,0.000,2011.0
buildingType,float64,2021,0,32,2053,0.634,0.000,0.01,0.644,0.048,4.0
constructionTime,object,0,19283,0,19283,0.000,6.048,0.00,6.048,0.000,2016.0
elevator,float64,32,0,0,32,0.010,0.000,0.00,0.010,0.000,1.0
fiveYearsProperty,float64,32,0,0,32,0.010,0.000,0.00,0.010,0.000,1.0


Rows with at least one problem: 166,585 (52.25% of 318,851)


As observed, the following columns have missing data - DOM, livingRoom, buildingType, elevator, fiveYearsProperty, subway, communityAverage.
the following columns have text where it is not expected - drawingRoom, constructionTime
and the following column has fraction where it expects int64 - buildingType.

More worringly, DOM has nearly half the data with NaN values. We should avoid utilising this variable.
Among the others rows, the nett % of bad rows are fairly small, dropping them should not influence bias significantly.

In [14]:
print(df.describe(include="object"))  # count/unique/top for text columns

/var/folders/hd/rg1pd_9n27l1j1s6_8hcl6rm0000gn/T/ipykernel_60231/1020512373.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.describe(include="object"))  # count/unique/top for text columns


                                                      url            id  \
count                                              318851        318851   
unique                                             318851        318851   
top     https://bj.lianjia.com/chengjiao/101084782030....  101084782030   
freq                                                    1             1   

        drawingRoom  bathRoom     floor  constructionTime  
count        318851    318851    318851            318851  
unique           22        18       169                74  
top               1         1  Middle 6              2004  
freq         225659    261488     34794             21145  


In [17]:
print(df.duplicated().sum())                # duplicate rows

0


In [18]:
print(df.nunique())                         # distinct values per column

url                    318851
id                     318851
Lng                      3995
Lat                      3993
Cid                      4035
tradeTime                2560
DOM                       551
followers                 547
totalPrice               5780
price                   80042
square                  18298
livingRoom                 10
drawingRoom                22
kitchen                     5
bathRoom                   18
floor                     169
buildingType               12
constructionTime           74
renovationCondition         5
buildingStructure           7
ladderRatio               183
elevator                    2
fiveYearsProperty           2
subway                      2
district                   13
communityAverage         4072
dtype: int64
